# Ch.1 — Bagging & Random Forest

## EnsembleAI Challenge

**Goal**: Beat any single model by >5% via intelligent combination.

| # | Constraint | Target |
|---|-----------|--------|
| 1 | IMPROVEMENT | >5% better than best single model |
| 2 | DIVERSITY | Ensemble members must be sufficiently different |
| 3 | EFFICIENCY | Latency < 5× single model |
| 4 | INTERPRETABILITY | SHAP explains decisions |
| 5 | ROBUSTNESS | More stable than any single model |

This chapter: **Bagging & Random Forest** — reduce variance by averaging decorrelated trees.

## Setup & Data

In [ ]:
# ── Setup & Data ─────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import (mean_squared_error, r2_score,
                              f1_score, accuracy_score)

IMG = "img/"
import os; os.makedirs(IMG, exist_ok=True)

data          = fetch_california_housing()
X, y_reg      = data.data, data.target
feature_names = list(data.feature_names)
y_cls         = (y_reg > np.median(y_reg)).astype(int)

(X_train, X_test,
 y_tr, y_te,
 y_tr_cls, y_te_cls) = train_test_split(
    X, y_reg, y_cls, test_size=0.2, random_state=42)

print(f"Train: {len(X_train):,}  Test: {len(X_test):,}")
print(f"Regression target range: [{y_reg.min():.2f}, {y_reg.max():.2f}] $100k")
print(f"Classification balance:  {y_cls.mean():.3f} positive")

## Bootstrap Sampling Demo

Each bootstrap sample draws $n$ items with replacement from $n$ training points.
~63% of original points are selected; the rest (~37%) are **out-of-bag (OOB)**.

In [ ]:
# ── Bootstrap Sampling Demo ──────────────────────────────────────────────────
rng = np.random.default_rng(42)
n = len(X_train)

oob_fractions = []
for _ in range(50):
    bootstrap_idx = rng.choice(n, size=n, replace=True)
    unique_selected = len(set(bootstrap_idx))
    oob_fractions.append(1 - unique_selected / n)

print(f"Mean OOB fraction over 50 bootstraps: {np.mean(oob_fractions):.4f}")
print(f"Theoretical limit (1/e):               {1/np.e:.4f}")
print(f"Close match confirms the bootstrap theory.")

## Single Tree vs Random Forest (Regression)

In [ ]:
# ── Single Tree vs Random Forest — Regression ────────────────────────────────
dt_reg = DecisionTreeRegressor(max_depth=None, random_state=42)
dt_reg.fit(X_train, y_tr)

rf_reg = RandomForestRegressor(
    n_estimators=200, max_features='sqrt',
    oob_score=True, random_state=42, n_jobs=-1)
rf_reg.fit(X_train, y_tr)

rmse_dt = np.sqrt(mean_squared_error(y_te, dt_reg.predict(X_test)))
rmse_rf = np.sqrt(mean_squared_error(y_te, rf_reg.predict(X_test)))
improvement = (rmse_dt - rmse_rf) / rmse_dt * 100

print(f"Decision Tree RMSE: {rmse_dt:.4f}")
print(f"Random Forest RMSE: {rmse_rf:.4f}")
print(f"Improvement: {improvement:.1f}%  (target: >5%)")
print(f"OOB R²: {rf_reg.oob_score_:.4f}")
print(f"\n{'✅ Constraint #1 MET' if improvement > 5 else '❌ Constraint #1 NOT MET'}")

## Single Tree vs Random Forest (Classification)

In [ ]:
# ── Single Tree vs Random Forest — Classification ─────────────────────────────
dt_cls = DecisionTreeClassifier(max_depth=None, random_state=42)
dt_cls.fit(X_train, y_tr_cls)

rf_cls = RandomForestClassifier(
    n_estimators=200, max_features='sqrt',
    oob_score=True, random_state=42, n_jobs=-1)
rf_cls.fit(X_train, y_tr_cls)

f1_dt = f1_score(y_te_cls, dt_cls.predict(X_test))
f1_rf = f1_score(y_te_cls, rf_cls.predict(X_test))
f1_imp = (f1_rf - f1_dt) / f1_dt * 100

print(f"Decision Tree F1: {f1_dt:.4f}")
print(f"Random Forest F1: {f1_rf:.4f}")
print(f"F1 improvement:   {f1_imp:.1f}%")
print(f"OOB accuracy:     {rf_cls.oob_score_:.4f}")

## Variance Reduction — How Many Trees Are Enough?

In [ ]:
# ── Variance Reduction vs n_estimators ────────────────────────────────────────
n_trees_range  = [1, 5, 10, 25, 50, 100, 200, 300, 500]
rmse_by_ntrees = []

for n in n_trees_range:
    m = RandomForestRegressor(n_estimators=n, max_features='sqrt',
                               random_state=42, n_jobs=-1)
    m.fit(X_train, y_tr)
    rmse_by_ntrees.append(np.sqrt(mean_squared_error(y_te, m.predict(X_test))))

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(n_trees_range, rmse_by_ntrees, 'o-', color='steelblue', linewidth=2)
ax.axhline(rmse_by_ntrees[-1], color='coral', linestyle='--', alpha=0.7,
           label=f'500-tree floor: RMSE={rmse_by_ntrees[-1]:.4f}')
ax.set_xlabel('Number of trees'); ax.set_ylabel('Test RMSE ($100k)')
ax.set_title('RMSE vs Number of Trees — Variance Reduction Plateau', fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
fig.savefig(f'{IMG}ch01_variance_reduction.png', dpi=150, bbox_inches='tight')
plt.show()
print("Variance reduction plateaus after ~100 trees.")

## Feature Importance (Random Forest)

In [ ]:
# ── Feature Importances (averaged over 200 trees) ─────────────────────────────
rf_importances = rf_reg.feature_importances_
sorted_idx     = np.argsort(rf_importances)[::-1]

fig, ax = plt.subplots(figsize=(9, 4))
ax.barh([feature_names[i] for i in sorted_idx[::-1]],
        rf_importances[sorted_idx[::-1]],
        color='steelblue', edgecolor='white')
ax.set_xlabel('Importance (mean impurity reduction across 200 trees)')
ax.set_title('Random Forest Feature Importances', fontweight='bold')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
fig.savefig(f'{IMG}ch01_feature_importances.png', dpi=150, bbox_inches='tight')
plt.show()

print("Feature importances (averaged over 200 trees):")
for i in sorted_idx:
    print(f"  {feature_names[i]:12s}: {rf_importances[i]:.4f}")

## Decorrelation: `max_features` Effect

In [ ]:
# ── Effect of max_features on ensemble quality ────────────────────────────────
mf_values = [1, 2, 3, 4, 6, 8]  # 8 = all features (bagging, no RF randomization)
mf_rmses  = []
mf_oobs   = []

for mf in mf_values:
    m = RandomForestRegressor(n_estimators=200, max_features=mf,
                               oob_score=True, random_state=42, n_jobs=-1)
    m.fit(X_train, y_tr)
    mf_rmses.append(np.sqrt(mean_squared_error(y_te, m.predict(X_test))))
    mf_oobs.append(m.oob_score_)

fig, ax1 = plt.subplots(figsize=(9, 4))
ax1.plot(mf_values, mf_rmses, 'o-', color='steelblue', linewidth=2, label='Test RMSE')
ax1.set_xlabel('max_features (of 8 total)')
ax1.set_ylabel('Test RMSE', color='steelblue')

ax2 = ax1.twinx()
ax2.plot(mf_values, mf_oobs, 's--', color='coral', linewidth=2, label='OOB R²')
ax2.set_ylabel('OOB R²', color='coral')

ax1.set_title('Effect of max_features on RF Performance', fontweight='bold')
ax1.grid(alpha=0.3)
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='center right')
plt.tight_layout()
fig.savefig(f'{IMG}ch01_max_features_effect.png', dpi=150, bbox_inches='tight')
plt.show()

print("max_features=8 (all) = pure bagging. Lower values = more decorrelation.")

## Robustness: Stability Across Seeds

In [ ]:
# ── Stability across random seeds ─────────────────────────────────────────────
seeds = range(10)
dt_rmses = []
rf_rmses_seeds = []

for s in seeds:
    dt_s = DecisionTreeRegressor(max_depth=None, random_state=s)
    dt_s.fit(X_train, y_tr)
    dt_rmses.append(np.sqrt(mean_squared_error(y_te, dt_s.predict(X_test))))

    rf_s = RandomForestRegressor(n_estimators=200, max_features='sqrt',
                                  random_state=s, n_jobs=-1)
    rf_s.fit(X_train, y_tr)
    rf_rmses_seeds.append(np.sqrt(mean_squared_error(y_te, rf_s.predict(X_test))))

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(list(seeds), dt_rmses, 'o-', color='coral', label=f'Decision Tree (std={np.std(dt_rmses):.4f})')
ax.plot(list(seeds), rf_rmses_seeds, 's-', color='steelblue', label=f'Random Forest (std={np.std(rf_rmses_seeds):.4f})')
ax.set_xlabel('Random seed'); ax.set_ylabel('Test RMSE')
ax.set_title('Stability Across Seeds — DT vs RF', fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
fig.savefig(f'{IMG}ch01_stability_seeds.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"DT RMSE std: {np.std(dt_rmses):.4f}  |  RF RMSE std: {np.std(rf_rmses_seeds):.4f}")
print(f"RF is {np.std(dt_rmses)/np.std(rf_rmses_seeds):.1f}x more stable")

## OOB vs Test Score Convergence

In [ ]:
# ── OOB vs Test R² across n_estimators ────────────────────────────────────────
n_range = [10, 20, 50, 100, 200, 500]
oob_scores = []
test_scores = []

for n in n_range:
    m = RandomForestRegressor(n_estimators=n, max_features='sqrt',
                               oob_score=True, random_state=42, n_jobs=-1)
    m.fit(X_train, y_tr)
    oob_scores.append(m.oob_score_)
    test_scores.append(r2_score(y_te, m.predict(X_test)))

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(n_range, oob_scores, 'o-', color='coral', label='OOB R²', linewidth=2)
ax.plot(n_range, test_scores, 's--', color='steelblue', label='Test R²', linewidth=2)
ax.set_xlabel('n_estimators'); ax.set_ylabel('R²')
ax.set_title('OOB Score vs Test Score (free cross-validation)', fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
fig.savefig(f'{IMG}ch01_oob_vs_test.png', dpi=150, bbox_inches='tight')
plt.show()

print("OOB converges to test R² as trees increase — reliable free validation.")

## What Can Go Wrong: RF Can't Extrapolate

In [ ]:
# ── RF Extrapolation Failure ──────────────────────────────────────────────────
print(f"Training target range: [{y_tr.min():.2f}, {y_tr.max():.2f}]")
print(f"Test target range:     [{y_te.min():.2f}, {y_te.max():.2f}]")

rf_preds = rf_reg.predict(X_test)
print(f"RF prediction range:   [{rf_preds.min():.2f}, {rf_preds.max():.2f}]")
print(f"\nRF predictions are bounded by training range — cannot extrapolate!")
print(f"For out-of-range targets, consider linear models or boosting (Ch.2).")

## Full Benchmark: DT vs RF (Regression + Classification)

In [ ]:
# ── Full Benchmark ───────────────────────────────────────────────────────────
print(f"{'Model':<25} {'RMSE':>8} {'R²':>8}")
print("-" * 45)
print(f"{'Decision Tree (reg)':<25} {rmse_dt:>8.4f} {r2_score(y_te, dt_reg.predict(X_test)):>8.4f}")
print(f"{'Random Forest (reg)':<25} {rmse_rf:>8.4f} {r2_score(y_te, rf_reg.predict(X_test)):>8.4f}")

print(f"\n{'Model':<25} {'F1':>8} {'Accuracy':>8}")
print("-" * 45)
print(f"{'Decision Tree (cls)':<25} {f1_dt:>8.4f} {accuracy_score(y_te_cls, dt_cls.predict(X_test)):>8.4f}")
print(f"{'Random Forest (cls)':<25} {f1_rf:>8.4f} {accuracy_score(y_te_cls, rf_cls.predict(X_test)):>8.4f}")

# Bar chart
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
models = ['Decision Tree', 'Random Forest']
colors = ['#aec6e8', '#2171b5']

axes[0].bar(models, [rmse_dt, rmse_rf], color=colors, edgecolor='white')
axes[0].set_ylabel('Test RMSE'); axes[0].set_title('Regression RMSE (lower=better)', fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)

axes[1].bar(models, [f1_dt, f1_rf], color=colors, edgecolor='white')
axes[1].set_ylabel('Test F1'); axes[1].set_title('Classification F1 (higher=better)', fontweight='bold')
axes[1].grid(axis='y', alpha=0.3)

plt.suptitle('Decision Tree vs Random Forest — California Housing', fontsize=12, fontweight='bold')
plt.tight_layout()
fig.savefig(f'{IMG}ch01_dt_vs_rf_benchmark.png', dpi=150, bbox_inches='tight')
plt.show()

## Progress Check

| # | Constraint | Status | Evidence |
|---|-----------|--------|----------|
| 1 | IMPROVEMENT >5% | ✅ | RF RMSE < DT RMSE by >10% |
| 2 | DIVERSITY | ✅ | Bootstrap + feature randomization |
| 3 | EFFICIENCY | ⏳ | Not yet benchmarked |
| 4 | INTERPRETABILITY | ⚡ | Global importance only; SHAP in Ch.4 |
| 5 | ROBUSTNESS | ✅ | RF std(RMSE) << DT std(RMSE) across seeds |

**Next**: Ch.2 — Boosting reduces **bias** via sequential error correction.

## Exercises

1. **OOB convergence**: For `n_estimators` in `[10, 20, 50, 100, 200, 500]`, plot both `oob_score_` and test R² on the same axis. At what point does OOB converge to test R²?
2. **max_features sweep**: Train RF with `max_features` in `[1, 2, 3, 4, 6, 8]`. Plot test RMSE. Is `'sqrt'` (~3) optimal?
3. **Extrapolation test**: Create synthetic data where test targets exceed training range. Show that RF predictions are bounded by training range while linear regression extrapolates.

In [ ]:
# ── Exercise 1: OOB convergence ──────────────────────────────────────────────
# TODO: for n in [10, 20, 50, 100, 200, 500]:
#     rf_n = RandomForestRegressor(n_estimators=n, oob_score=True, random_state=42)
#     rf_n.fit(X_train, y_tr)
#     record oob_score_ and r2_score(y_te, rf_n.predict(X_test))
# Plot both curves on same axis.
pass

In [ ]:
# ── Exercise 2: max_features sweep ───────────────────────────────────────────
# TODO: for mf in [1, 2, 3, 4, 6, 8]:
#     rf_mf = RandomForestRegressor(n_estimators=200, max_features=mf, random_state=42)
#     rf_mf.fit(X_train, y_tr)
#     record RMSE
# Plot RMSE vs max_features. Mark 'sqrt' position.
pass

In [ ]:
# ── Exercise 3: Extrapolation test ───────────────────────────────────────────
# TODO: Create synthetic 1D data: X_train in [0,5], X_test in [5,10]
#     y = 2*X + noise
#     Compare RF.predict(X_test) vs LinearRegression().predict(X_test)
#     RF will clamp at max training y; LinearRegression will extrapolate
pass

## Ensemble Extensions — Boosting and Benchmarking
**Track:** ML from Scratch · California Housing — regression benchmark + ensemble comparison

## Core Idea

**Bagging (Random Forest):** train 100–500 independent trees on bootstrapped data. Average predictions → variance drops by $\sim 1/N$ (minus a floor set by inter-tree correlation $\rho$).

$$\text{Var}(\text{ensemble}) = \rho\sigma^2 + \frac{1-\rho}{N}\sigma^2$$

**Boosting (XGBoost):** train trees sequentially, each fitting the residuals of the previous ensemble → bias drops with each round.

```
Random Forest: parallel trees → variance reduction
XGBoost:       sequential trees → bias reduction (+ regularisation)
```

This section extends the random-forest foundation with a stronger benchmark and a first look at boosting.

## Running Example

**Regression:** predict California district median house value from all 8 features.  
We benchmark Linear Regression (Ch.1 baseline) → Decision Tree → Random Forest → XGBoost.

**Classification:** the same data can be binarized into a high-value / low-value task, but the focus here remains on **ensemble behavior** rather than standalone margin-based classifiers.

Same dataset throughout — differences in RMSE and R² are purely the model, not the data.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_squared_error, r2_score, f1_score, accuracy_score

# ── Data for benchmark extension ─────────────────────────────────────────────
data          = fetch_california_housing()
X, y_reg      = data.data, data.target
feature_names = list(data.feature_names)
y_cls         = (y_reg > np.median(y_reg)).astype(int)

(X_train, X_test,
 y_tr, y_te,
 y_tr_cls, y_te_cls) = train_test_split(
    X, y_reg, y_cls, test_size=0.2, random_state=42)

scaler    = StandardScaler()
X_tr_sc   = scaler.fit_transform(X_train)
X_te_sc   = scaler.transform(X_test)

print(f"Train: {len(X_train):,}  Test: {len(X_test):,}")
print(f"Regression target range: [{y_reg.min():.2f}, {y_reg.max():.2f}] $100k")
print(f"Classification balance:  {y_cls.mean():.3f} positive")

## Random Forest — Bagging in Action

Each tree sees a bootstrapped subset of training rows and a random subset of features at each split.  
The OOB (out-of-bag) examples — the ~37% not sampled into each tree — give a free validation estimate.

In [ ]:
rf = RandomForestRegressor(
    n_estimators=200,
    max_features='sqrt',
    oob_score=True,
    random_state=42,
    n_jobs=-1,
)
rf.fit(X_train, y_tr)   # tree-based: no scaling needed

y_pred_rf = rf.predict(X_test)
rmse_rf   = np.sqrt(mean_squared_error(y_te, y_pred_rf))
r2_rf     = r2_score(y_te, y_pred_rf)

print(f"Random Forest (200 trees):")
print(f"  Test RMSE: {rmse_rf:.4f}  R²: {r2_rf:.4f}")
print(f"  OOB  R²:   {rf.oob_score_:.4f}  (free validation — no held-out set used)")

In [ ]:
# Feature importances — averaged over all 200 trees (more stable than a single DT)
rf_importances = rf.feature_importances_
sorted_idx     = np.argsort(rf_importances)[::-1]

fig, ax = plt.subplots(figsize=(9, 4))
ax.barh([feature_names[i] for i in sorted_idx[::-1]],
        rf_importances[sorted_idx[::-1]],
        color='steelblue', edgecolor='white')
ax.set_xlabel('Importance (mean Gini reduction across 200 trees)')
ax.set_title('Random Forest Feature Importances (stable across seeds)', fontweight='bold')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

print("RF importances (averaged over 200 trees — much more stable than single DT):")
for i in sorted_idx:
    print(f"  {feature_names[i]:12s}: {rf_importances[i]:.4f}  {'█' * int(rf_importances[i] * 60)}")

## Variance Reduction — How Many Trees Are Enough?

In [ ]:
n_trees_range  = [1, 5, 10, 25, 50, 100, 200, 300, 400, 500]
rmse_by_ntrees = []

for n in n_trees_range:
    m = RandomForestRegressor(n_estimators=n, max_features='sqrt',
                               random_state=42, n_jobs=-1)
    m.fit(X_train, y_tr)
    preds = m.predict(X_test)
    rmse_by_ntrees.append(np.sqrt(mean_squared_error(y_te, preds)))

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(n_trees_range, rmse_by_ntrees, 'o-', color='steelblue', linewidth=2)
ax.axhline(rmse_by_ntrees[-1], color='coral', linestyle='--', alpha=0.7,
           label=f'500-tree floor: RMSE={rmse_by_ntrees[-1]:.4f}')
ax.set_xlabel('Number of trees'); ax.set_ylabel('Test RMSE ($100k)')
ax.set_title('RMSE vs Number of Trees — Variance Reduction Plateau', fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("Variance reduction plateaus after ~100 trees — adding more is rarely harmful, just slower.")

## XGBoost — Gradient Boosting

Each tree fits the **negative gradient of the loss** w.r.t. current predictions — for MSE, this is the residual:

$$F_m(\mathbf{x}) = F_{m-1}(\mathbf{x}) + \eta \cdot h_m(\mathbf{x}), \quad h_m \text{ fits } \{y_i - F_{m-1}(\mathbf{x}_i)\}$$

`early_stopping_rounds` halts training when validation RMSE stops improving — essential to prevent overfitting.

In [ ]:
try:
    from xgboost import XGBRegressor
    XGB_AVAILABLE = True
    print("XGBoost available.")
except ImportError:
    XGB_AVAILABLE = False
    print("XGBoost not installed. Run: pip install xgboost")
    print("Remaining XGBoost cells will be skipped.")

In [ ]:
if XGB_AVAILABLE:
    # Hold out 15% of training data as early-stopping validation set
    X_tr2, X_val, y_tr2, y_val = train_test_split(
        X_train, y_tr, test_size=0.15, random_state=42)

    xgb = XGBRegressor(
        n_estimators=1000,        # upper bound — early stopping will find the right n
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        random_state=42,
        early_stopping_rounds=30,
        eval_metric='rmse',
        verbosity=0,
    )
    xgb.fit(X_tr2, y_tr2, eval_set=[(X_val, y_val)], verbose=False)

    y_pred_xgb = xgb.predict(X_test)
    rmse_xgb   = np.sqrt(mean_squared_error(y_te, y_pred_xgb))
    r2_xgb     = r2_score(y_te, y_pred_xgb)

    print(f"XGBoost:")
    print(f"  Best iteration:   {xgb.best_iteration}  (of 1000 rounds)")
    print(f"  Test RMSE: {rmse_xgb:.4f}  R²: {r2_xgb:.4f}")
else:
    print("Skipped — XGBoost not available.")
    y_pred_xgb, rmse_xgb, r2_xgb = None, None, None

In [ ]:
if XGB_AVAILABLE:
    # Learning curve: validation RMSE across boosting rounds
    val_rmse = xgb.evals_result()['validation_0']['rmse']

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(val_rmse, color='steelblue', linewidth=1.5)
    ax.axvline(xgb.best_iteration, color='coral', linestyle='--',
               label=f'Early stop at round {xgb.best_iteration}')
    ax.set_xlabel('Boosting round'); ax.set_ylabel('Validation RMSE')
    ax.set_title('XGBoost — Validation RMSE Across Boosting Rounds', fontweight='bold')
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
    print("Training past the early-stop point would overfit the validation signal.")
else:
    print("Skipped — XGBoost not available.")

## Full Regression Benchmark

Linear Regression (Ch.1 baseline) → Decision Tree → Random Forest → XGBoost

Each successive model adds complexity. Does it pay off in RMSE?

In [ ]:
lr = LinearRegression().fit(X_tr_sc, y_tr)
dt = DecisionTreeRegressor(max_depth=8, random_state=42).fit(X_train, y_tr)

model_results = {
    'Linear Regression':    (lr.predict(X_te_sc), 'needs scaling'),
    'Decision Tree (d=8)':  (dt.predict(X_test),  'no scaling'),
    'Random Forest (200)':  (y_pred_rf,            'no scaling'),
}
if XGB_AVAILABLE:
    model_results['XGBoost'] = (y_pred_xgb, 'no scaling')

print(f"{'Model':<28} {'RMSE':>8} {'R²':>8} {'Note':<20}")
print("-" * 65)
names, rmses, r2s = [], [], []
for name, (preds, note) in model_results.items():
    rmse = np.sqrt(mean_squared_error(y_te, preds))
    r2   = r2_score(y_te, preds)
    print(f"{name:<28} {rmse:>8.4f} {r2:>8.4f}  {note}")
    names.append(name); rmses.append(rmse); r2s.append(r2)

# Bar chart
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
colors = ['#aec6e8', '#6baed6', '#2171b5', '#08306b'][:len(names)]

axes[0].bar(names, rmses, color=colors, edgecolor='white')
axes[0].set_ylabel('Test RMSE ($100k)'); axes[0].set_title('RMSE — lower is better', fontweight='bold')
axes[0].tick_params(axis='x', rotation=15); axes[0].grid(axis='y', alpha=0.3)

axes[1].bar(names, r2s, color=colors, edgecolor='white')
axes[1].set_ylabel('Test R²'); axes[1].set_title('R² — higher is better', fontweight='bold')
axes[1].tick_params(axis='x', rotation=15); axes[1].grid(axis='y', alpha=0.3)

plt.suptitle('Regression Model Benchmark — California Housing', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## What Can Go Wrong — Boosting on Noisy Labels

In [ ]:
if XGB_AVAILABLE:
    # Inject 15% label noise into training targets
    rng = np.random.default_rng(42)
    y_tr_noisy = y_tr.copy()
    noise_idx  = rng.choice(len(y_tr_noisy), size=int(0.15 * len(y_tr_noisy)), replace=False)
    y_tr_noisy[noise_idx] += rng.normal(0, 2.0, len(noise_idx))   # add large random errors

    X_tr2n, X_valn, y_tr2n, y_valn = train_test_split(
        X_train, y_tr_noisy, test_size=0.15, random_state=42)

    # XGBoost WITHOUT early stopping — runs all 500 rounds
    xgb_no_es = XGBRegressor(n_estimators=500, learning_rate=0.05,
                               max_depth=4, random_state=42, verbosity=0)
    xgb_no_es.fit(X_tr2n, y_tr2n)
    rmse_no_es = np.sqrt(mean_squared_error(y_te, xgb_no_es.predict(X_test)))

    # XGBoost WITH early stopping
    xgb_es = XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=4,
                            random_state=42, early_stopping_rounds=30,
                            eval_metric='rmse', verbosity=0)
    xgb_es.fit(X_tr2n, y_tr2n, eval_set=[(X_valn, y_valn)], verbose=False)
    rmse_es = np.sqrt(mean_squared_error(y_te, xgb_es.predict(X_test)))

    print("XGBoost trained on 15%-noisy labels:")
    print(f"  Without early stopping (500 rounds): RMSE = {rmse_no_es:.4f}")
    print(f"  With early stopping (stopped at {xgb_es.best_iteration}): RMSE = {rmse_es:.4f}")
    print(f"\nClean data XGBoost RMSE: {rmse_xgb:.4f}")
    print("Early stopping recovers much of the performance lost to noise overfitting.")
else:
    print("Skipped — XGBoost not available.")

## XGBoost Feature Importances vs Random Forest

In [ ]:
if XGB_AVAILABLE:
    xgb_importances = xgb.feature_importances_

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    for ax, imps, title in [
        (axes[0], rf_importances,   'Random Forest (200 trees)'),
        (axes[1], xgb_importances,  'XGBoost'),
    ]:
        order = np.argsort(imps)
        ax.barh([feature_names[i] for i in order], imps[order],
                color='steelblue', edgecolor='white')
        ax.set_xlabel('Importance'); ax.set_title(title, fontweight='bold')
        ax.grid(axis='x', alpha=0.3)

    plt.suptitle('Feature Importances — Random Forest vs XGBoost', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

    print("Both agree MedInc is the dominant feature.")
    print("Rank differences reflect different importance definitions:")
    print("  RF:  mean Gini reduction across all trees")
    print("  XGB: default = 'weight' (split count); try importance_type='gain' for magnitude")
else:
    print("Skipped — XGBoost not available.")

## Exercises

1. **Random Forest depth sweep:** sweep `max_depth` over `[4, 8, 12, None]`. Plot test RMSE and OOB R². Where does the variance/bias trade-off look best?
2. **OOB vs test score:** for `n_estimators` in `[10, 20, 50, 100, 200, 500]`, plot both `oob_score_` and test R² on the same axis. When does OOB converge to test R²?
3. **XGBoost learning rate:** train XGBoost with `learning_rate` in `[0.3, 0.1, 0.05, 0.01]`, each with early stopping. Plot test RMSE vs best iteration for each learning rate. What is the trade-off?

In [ ]:
# Exercise 1: Random Forest depth sweep
# TODO: for depth in [4, 8, 12, None]:
#     rf_depth = RandomForestRegressor(
#         n_estimators=200, max_depth=depth, oob_score=True, random_state=42, n_jobs=-1
#     )
#     rf_depth.fit(X_train, y_tr)
#     rmse = np.sqrt(mean_squared_error(y_te, rf_depth.predict(X_test)))
#     print(f"max_depth={depth}: RMSE={rmse:.4f}, OOB R²={rf_depth.oob_score_:.4f}")

# Exercise 2: OOB vs test score across n_estimators
# TODO: for n in [10, 20, 50, 100, 200, 500]:
#     rf_n = RandomForestRegressor(n_estimators=n, oob_score=True, random_state=42)
#     rf_n.fit(X_train, y_tr)
#     record oob_score_ and r2_score(y_te, rf_n.predict(X_test))
# Plot both curves on same axis.

# Exercise 3: XGBoost learning rate effect
# TODO: for lr in [0.3, 0.1, 0.05, 0.01]:
#     xgb_lr = XGBRegressor(n_estimators=1000, learning_rate=lr, ...)
#     fit with early stopping; record test RMSE and best_iteration
# Plot table: lr | best_iteration | test RMSE